In [2]:
import sys
sys.path.append("..")

In [3]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [4]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [10]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)
    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, i, x_0, theta_0, x_r)

    df_results = pd.DataFrame(results)
    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f'../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl')
    
    return df_results

In [11]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)

        # rng = np.random.default_rng(seed=seed)
        # size_N = int(np.rint(0.15 * recourse_needed_X_test.shape[0]))
        # recourse_needed_X_test = rng.choice(recourse_needed_X_test, size=size_N, replace=False) 
        
        for recourse_fn in recourse_fns:
            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

In [15]:
alphas = np.linspace(0,0.5,5)
lambdas = [0.1, 0.3]

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True

        datasets = [SBADataset()]
        recourse_fns = [LARRecourse]

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            # ret = run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running sba data...


[Alg1] [alpha=0.0] [lambda=0.1]: 100%|██████████| 39/39 [00:00<00:00, 4492.91it/s]


[Alg1] Saving results for sba run 0


[Alg1] [alpha=0.0] [lambda=0.1]: 100%|██████████| 36/36 [00:00<00:00, 4486.82it/s]


[Alg1] Saving results for sba run 1


...

[Alg1] Saving results for sba run 2


[Alg1] [alpha=0.5] [lambda=0.3]: 100%|██████████| 36/36 [00:00<00:00, 7541.83it/s]


[Alg1] Saving results for sba run 3


[Alg1] [alpha=0.5] [lambda=0.3]: 100%|██████████| 38/38 [00:00<00:00, 6970.02it/s]


[Alg1] Saving results for sba run 4
Finished sba



In [ ]:
# torch.manual_seed(0)

# d_results = {}
# params = {}
# params['alpha'] = 0.5 # float, None
# params['lamb'] = 0.1
# params['seeds'] = [2]
# params['save_results'] = True

# datasets = [SBADataset()]
# recourse_fns = [L1Recourse]

# for dataset in datasets:
#     results = []
#     print(f'Running {dataset.name} data...')
#     run_experiment(dataset, recourse_fns, params, results)
#     # ret = run_experiment(dataset, recourse_fns, params, results)
    
#     d_results[dataset.name] = pd.concat(results)
#     print(f'Finished {dataset.name}\n')

Running sba data...
Finished sba

